# Week 4-3 — LinearRegression과 RandomForest 비교

Feature set B를 고정하고 persistence baseline, LinearRegression, RandomForest를 동일한 validation 행에서 비교한다.

In [234]:
import pandas as pd
import numpy as np

ds = pd.read_csv('../../../data/week4_korea_cli.csv')
ds

,observation_date,KORLOLITOAASTSAM
0,1990-01-01,100.461861
1,1990-02-01,100.462284
2,1990-03-01,100.536374
3,1990-04-01,100.613811
4,1990-05-01,100.646672
...,...,...
433,2026-02-01,101.597894
434,2026-03-01,102.001885
435,2026-04-01,102.363604
436,2026-05-01,102.657914


In [235]:
ds.observation_date = pd.to_datetime(ds.observation_date)
ds = ds.sort_values('observation_date')

# TODO C1: 날짜가 실제로 오름차순인지 확인하는 결과를 출력하세요.
ds.observation_date.is_monotonic_increasing

True

In [236]:
ds['target_next_month'] = ds['KORLOLITOAASTSAM'].shift(-1).copy()
ds['baseline'] = ds['KORLOLITOAASTSAM'].copy()
ds['cli_lag1'] = ds['KORLOLITOAASTSAM'].shift(1).copy()
ds['cli_rolling3'] = ds['KORLOLITOAASTSAM'].rolling(window=3).mean()
ds['cli_diff1'] = ds['KORLOLITOAASTSAM'].diff(1)

In [237]:
set_B = ['cli_lag1', 'cli_rolling3', 'cli_diff1']

In [238]:
ds = ds.dropna(
    subset=['cli_lag1', 'cli_rolling3', 'cli_diff1',
            'KORLOLITOAASTSAM', 'target_next_month']
)

ds

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3,cli_diff1
2,1990-03-01,100.536374,100.613811,100.536374,100.462284,100.486840,0.074089
3,1990-04-01,100.613811,100.646672,100.613811,100.536374,100.537490,0.077438
4,1990-05-01,100.646672,100.610137,100.646672,100.613811,100.598952,0.032861
5,1990-06-01,100.610137,100.477904,100.610137,100.646672,100.623540,-0.036535
6,1990-07-01,100.477904,100.254061,100.477904,100.610137,100.578238,-0.132233
...,...,...,...,...,...,...,...
432,2026-01-01,101.190196,101.597894,101.190196,100.813483,100.830936,0.376713
433,2026-02-01,101.597894,102.001885,101.597894,101.190196,101.200525,0.407698
434,2026-03-01,102.001885,102.363604,102.001885,101.597894,101.596658,0.403991
435,2026-04-01,102.363604,102.657914,102.363604,102.001885,101.987795,0.361719


In [239]:
train = ds[
    (ds.observation_date >= pd.to_datetime('2000-01-01')) &
    (ds.observation_date <= pd.to_datetime('2014-12-01'))
].copy()

validation = ds[
    (ds.observation_date >= pd.to_datetime('2015-01-01')) &
    (ds.observation_date <= pd.to_datetime('2019-12-01'))
].copy()

In [240]:
print('train rows:', len(train))
print('train dates:', train.observation_date.min(), train.observation_date.max())
print('validation rows:', len(validation))
print('validation dates:', validation.observation_date.min(), validation.observation_date.max())

train rows: 180
train dates: 2000-01-01 00:00:00 2014-12-01 00:00:00
validation rows: 60
validation dates: 2015-01-01 00:00:00 2019-12-01 00:00:00


In [241]:
X_train_B = train[set_B].copy()
X_validation_B = validation[set_B].copy()
y_train = train['target_next_month'].copy()
y_validation = validation['target_next_month'].copy()

X_train_B.shape, X_validation_B.shape, X_train_B.columns.tolist()

((180, 3), (60, 3), ['cli_lag1', 'cli_rolling3', 'cli_diff1'])

## C2 — LinearRegression with feature set B

TODO: 아래 셀에서 feature set B를 사용하는 LinearRegression을 직접 생성하고, train에서 학습한 뒤 validation 예측을 만드세요.

In [242]:
from sklearn.linear_model import LinearRegression

# TODO C2: LinearRegression 객체 생성, fit, validation 예측
model_linear = LinearRegression()
model_linear.fit(X_train_B, y_train)
validation_predict = model_linear.predict(X_validation_B)

## C3 — RandomForestRegressor with feature set B

In [243]:
from sklearn.ensemble import RandomForestRegressor

random_forest_B = RandomForestRegressor(
    n_estimators=200,
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)

random_forest_B.fit(X_train_B, y_train)
random_forest_pred_B = random_forest_B.predict(X_validation_B)

# TODO C3: Random Forest validation 예측 배열의 type과 shape를 출력하세요.
type(random_forest_pred_B), random_forest_pred_B.shape

(numpy.ndarray, (60,))

In [244]:
random_forest_mae = (y_validation - random_forest_pred_B).abs().mean()
random_forest_rmse = np.sqrt(((y_validation - random_forest_pred_B) ** 2).mean())

random_forest_mae, random_forest_rmse

(np.float64(0.1946600051728891), np.float64(0.251602695748719))

## C4 — 공통 validation 비교표

TODO: `persistence_baseline`, `linear_regression_B`, `random_forest_B`의 `model`, `feature_set`, `validation_rows`, `MAE`, `RMSE`를 같은 표에 넣고 MAE 오름차순으로 정렬하세요.

In [245]:
result = pd.DataFrame(index=['persistence_baseline', 'linear_regression_B', 'random_forest_B'])

# TODO C4: 세 후보의 공통 validation 비교표를 직접 완성하세요.
result.loc['persistence_baseline', 'model'] = 'baseline'
result.loc['persistence_baseline', 'feature_set'] = 1
result.loc['persistence_baseline', 'validation_rows'] = len(validation)
result.loc['persistence_baseline', 'MAE'] = (validation.target_next_month - validation.baseline).abs().mean()
result.loc['persistence_baseline', 'RMSE'] = np.sqrt(((validation.target_next_month - validation.baseline)**2).mean())

result.loc['linear_regression_B', 'model'] = 'B_linear'
result.loc['linear_regression_B', 'feature_set'] = len(set_B)
result.loc['linear_regression_B', 'validation_rows'] = len(validation)
result.loc['linear_regression_B', 'MAE'] = (validation.target_next_month - validation_predict).abs().mean()
result.loc['linear_regression_B', 'RMSE'] = np.sqrt(((validation.target_next_month - validation_predict)**2).mean())


result.loc['random_forest_B', 'model'] = 'B_rf'
result.loc['random_forest_B', 'feature_set'] = len(set_B)
result.loc['random_forest_B', 'validation_rows'] = len(validation)
result.loc['random_forest_B', 'MAE'] = random_forest_mae
result.loc['random_forest_B', 'RMSE'] = random_forest_rmse

In [246]:
result = result.sort_values(by='MAE',ascending=True)
result

,model,feature_set,validation_rows,MAE,RMSE
linear_regression_B,B_linear,3.0,60.0,0.010768,0.013730
persistence_baseline,baseline,1.0,60.0,0.079752,0.093522
random_forest_B,B_rf,3.0,60.0,0.194660,0.251603
